In [2]:
import hoomd
import hoomd.md
import hoomd.write
import numpy as np
import matplotlib.pyplot as plt
import yaml
from pathlib import Path

In [3]:
project_root = Path.cwd().parent

In [4]:
def load_hydropathy(scale_name):
    yaml_path = project_root / "configs" / "simulation" / "hydropathy_scales.yaml"
    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f)
    return data[scale_name]

In [16]:
def load_configs(config_path="defaults.yaml"):
    yaml_path = project_root / "configs" / "simulation" / config_path

    with open(yaml_path, 'r') as f:
        config = yaml.safe_load(f)
    return config
config_path = project_root/"configs"/"run_config.yaml"
r = config_path.parent.parent
print(r)

/home/andres/Research/machine-learning-IDPs


In [12]:

AA_PARAMS = {
    "A": dict(mass= 71.08, charge= 0.0, sigma=0.504, HPS1=0.730, HPS2=0.003),
    "R": dict(mass=156.20, charge= 1.0, sigma=0.656, HPS1=0.000, HPS2=0.723),
    "N": dict(mass=114.10, charge= 0.0, sigma=0.568, HPS1=0.432, HPS2=0.160),
    "D": dict(mass=115.10, charge=-1.0, sigma=0.558, HPS1=0.378, HPS2=0.002),
    "C": dict(mass=103.10, charge= 0.0, sigma=0.548, HPS1=0.595, HPS2=0.400),
    "Q": dict(mass=128.10, charge= 0.0, sigma=0.602, HPS1=0.514, HPS2=0.468),
    "E": dict(mass=129.10, charge=-1.0, sigma=0.592, HPS1=0.459, HPS2=0.022),
    "G": dict(mass= 57.05, charge= 0.0, sigma=0.450, HPS1=0.649, HPS2=0.784),
    "H": dict(mass=137.10, charge= 0.5, sigma=0.608, HPS1=0.514, HPS2=0.487),
    "I": dict(mass=113.20, charge= 0.0, sigma=0.618, HPS1=0.973, HPS2=0.687),
    "L": dict(mass=113.20, charge= 0.0, sigma=0.618, HPS1=0.973, HPS2=0.335),
    "K": dict(mass=128.20, charge= 1.0, sigma=0.636, HPS1=0.514, HPS2=0.095),
    "M": dict(mass=131.20, charge= 0.0, sigma=0.618, HPS1=0.838, HPS2=0.993),
    "F": dict(mass=147.20, charge= 0.0, sigma=0.636, HPS1=1.000, HPS2=0.871),
    "P": dict(mass= 97.12, charge= 0.0, sigma=0.556, HPS1=1.000, HPS2=0.471),
    "S": dict(mass= 87.08, charge= 0.0, sigma=0.518, HPS1=0.595, HPS2=0.487),
    "T": dict(mass=101.10, charge= 0.0, sigma=0.562, HPS1=0.676, HPS2=0.274),
    "W": dict(mass=186.20, charge= 0.0, sigma=0.678, HPS1=0.946, HPS2=0.753),
    "Y": dict(mass=163.20, charge= 0.0, sigma=0.646, HPS1=0.865, HPS2=0.984),
    "V": dict(mass= 99.07, charge= 0.0, sigma=0.586, HPS1=0.892, HPS2=0.428),
}

# ── Derived arrays — single source of truth ───────────────────────────────────
AA_ORDER = list(AA_PARAMS.keys())
ids      = {aa: i for i, aa in enumerate(AA_ORDER)}

HYDROPATHY_SCALES = {"HPS1", "HPS2"}  # valid keys, for validation

def get_hps(scale: str) -> np.ndarray:
    if scale not in HYDROPATHY_SCALES:
        raise ValueError(f"Unknown hydropathy scale '{scale}'. Valid: {HYDROPATHY_SCALES}")
    return np.array([AA_PARAMS[aa][scale] for aa in AA_ORDER], dtype=np.float32)

lambdas = get_hps("HPS1")
masses  = np.array([AA_PARAMS[aa]["mass"]   for aa in AA_ORDER], dtype=np.float32)
charges = np.array([AA_PARAMS[aa]["charge"] for aa in AA_ORDER], dtype=np.float32)
sigmas  = np.array([AA_PARAMS[aa]["sigma"]  for aa in AA_ORDER], dtype=np.float32)

print(lambdas)

[0.73  0.    0.432 0.378 0.595 0.514 0.459 0.649 0.514 0.973 0.973 0.514
 0.838 1.    1.    0.595 0.676 0.946 0.865 0.892]


In [7]:
AA_ORDER = ['A', 'R', 'N', 'D', 'C', 'Q', 'E', 'G', 'H', 'I', 
            'L', 'K', 'M', 'F', 'P', 'S', 'T', 'W', 'Y', 'V']
hps_dict = load_hydropathy("HPS2")
missing = set(AA_ORDER) - set(hps_dict.keys())
if missing:
    raise ValueError(f"Hydropathy scale is missing values for: {missing}")
    
ids = {aa: i for i, aa in enumerate(AA_ORDER)}

hps = np.array([hps_dict[aa] for aa in AA_ORDER])

masses = np.array([71.08, 156.2, 114.1, 115.1, 103.1, 128.1, 129.1, 57.05, 137.1, 113.2
, 113.2, 128.2, 131.2, 147.2, 97.12, 87.08, 101.1, 186.2, 163.2, 99.07], dtype=np.float32)

charges = np.array([0.0, 1.0, 0.0, -1.0, 0.0, 0.0, -1.0, 0.0, 0.5, 0.0, 0.0, 1.0, 0.0, 0.0
, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], dtype=np.float32)

sigmas = np.array([0.504, 0.656, 0.568, 0.558, 0.548, 0.602, 0.592, 0.45, 0.608, 0.618, 0.618, 0.636
 , 0.618, 0.636, 0.556, 0.518, 0.562, 0.678, 0.646, 0.586], dtype=np.float32)

print(hps_dict)

{'A': 0.003, 'R': 0.723, 'N': 0.16, 'D': 0.002, 'C': 0.4, 'Q': 0.468, 'E': 0.022, 'G': 0.784, 'H': 0.487, 'I': 0.687, 'L': 0.335, 'K': 0.095, 'M': 0.993, 'F': 0.871, 'P': 0.471, 'S': 0.487, 'T': 0.274, 'W': 0.753, 'Y': 0.984, 'V': 0.428}


In [41]:
#
# MITTAL IDPS. REPLICATE SMALLER SEQUENCES IN PAPER 

# CHECK ANALYSISS FOLDER TO PRODUCE GSD MOVIE AND RG DISTRIBUTIONS 
#
# CHECK BOND LENGHT 
# check MD analysis library to analysis the trajectory(gsd) calculate stifness (average coss thetha) and produce distributions
# after everything is seutp the parameters to vary between runs will be sequence and PH 

# look at strange events in the distributions further from the average lenght 

# steps = 5(N^(2.2))/dt
# ── Settings ──────────────────────────────────────────────────────────────────
sequence  = 'GSHCFLDGIDKAQEEHEKYHSNWRAMASDFNLPPVVAKEIVASCDKCQLKGEAMHGQVDC'
N         = len(sequence)
GSD_FILE  = 'very_cold_run.gsd'
BOX_SIZE  = 500    # nm, appropriate for a 23-residue chain
T_Kelvin  = 5      #  ~room temp in HPS
DT        = 0.01     # timestep
N_STEPS   = int((5*(N)**2.2)/DT)
SAVE_EVERY= 1000      # save frame every N steps -> 100 frames total
EPS_HPS   = 0.2       # kJ/mol, fixed LJ epsilon for all pairs
# ──────────────────────────────────────────────────────────────────────────────
KT = T_Kelvin*0.001987204259 # Boltzmann constant in kCal/mol/K
EPSILON = 0.1 # KCal/mol

sqid = np.array([ids[aa] for aa in sequence], dtype=np.int32)

mass = masses[sqid]
charge = charges[sqid]
sigma = sigmas[sqid]
hp = hps[sqid]

ionic_concentration = 150*1e-3 # in M or mol/L

fepsw = lambda T : 5321/T+233.76-0.9297*T+0.1417*1e-2*T*T-0.8292*1e-6*T**3 #temperature dependent dielectric constant of water
epsw = fepsw(T_Kelvin) # dielectric constant of water at T 
lB = (1.6021766**2/(4*np.pi*8.854188*epsw))*(6.022*1000/KT)/4.184 # Bjerrum length in nm

yukawa_eps = lB*KT
yukawa_kappa = np.sqrt(8*np.pi*lB*ionic_concentration*6.022/10)



In [42]:
# ── Build snapshot ────────────────────────────────────────────────────────────
device = hoomd.device.auto_select()
sim = hoomd.Simulation(device=device, seed=42)

snapshot = hoomd.Snapshot()
snapshot.configuration.box = [BOX_SIZE, BOX_SIZE, BOX_SIZE, 0, 0, 0]
snapshot.particles.types = AA_ORDER
snapshot.particles.N = N
snapshot.bonds.N = N - 1      # must pre-allocate before populate_snapshot
snapshot.bonds.types = ['AA-bond']
# Particles
snapshot.particles.typeid[:] = sqid
snapshot.particles.mass[:]   = mass
snapshot.particles.charge[:] = charge
snapshot.particles.diameter[:] = sigma

# Place beads in a straight line with ~0.38 nm spacing (realistic Cα-Cα)
snapshot.particles.position[:] = [[i * 0.38 - (N * 0.38) / 2, 0, 0] for i in range(N)]

# Bonds: chain connectivity
snapshot.bonds.N = N - 1
snapshot.bonds.types = ['AA-bond']
snapshot.bonds.typeid[:] = [0] * (N - 1)
snapshot.bonds.group[:]  = [[i, i + 1] for i in range(N - 1)]

sim.create_state_from_snapshot(snapshot)

In [43]:
# ── Forces ────────────────────────────────────────────────────────────────────
nl = hoomd.md.nlist.Tree(buffer=0.2)
nl.exclusions = ['bond']
lj1 = hoomd.md.pair.LJ(nlist=nl, mode='shift')
lj2 = hoomd.md.pair.LJ(nlist=nl)


yukawa = hoomd.md.pair.Yukawa(nlist=nl, default_r_cut=3.5, mode='shift')

##
# CHECK MODEL IN PAPER
#
# HPS pair interactions: WCA (repulsive) if mean hydropathy < 0.5, else full LJ
for i, aa1 in enumerate(AA_ORDER):
    for j, aa2 in enumerate(AA_ORDER):
        if j < i:
            continue
        sig_ij = (sigma[i] + sigma[j]) / 2
        lam_ij = (hp[i]   + hp[j])   / 2
        lj2.params[(aa1, aa2)] = dict(epsilon=EPSILON*lam_ij, sigma=sig_ij)
        lj1.params[(aa1, aa2)] = dict(epsilon=EPSILON*(1-lam_ij), sigma=sig_ij)
        
        lj1.r_cut[(aa1, aa2)] = 2**(1/6) * sig_ij   # purely repulsive (WCA)

        lj2.r_cut[(aa1, aa2)] = 3.5        # attractive
        yukawa.params[(aa1, aa2)] = dict(epsilon=yukawa_eps*charge[i] * charge[j], kappa=yukawa_kappa)
        yukawa.r_cut[(aa1, aa2)]  = 3.5

# FENE bonds for chain connectivity

#------------------------------------------
# USE HARMONIC BONDS INSTEAD OF FENE
#-------------------------------------------
# fene = hoomd.md.bond.FENEWCA() 
# fene.params['AA-bond'] = dict(k=30.0, r0=1.5, epsilon=1.0, sigma=0.47, delta=0.0)

harmonic = hoomd.md.bond.Harmonic()
harmonic.params['AA-bond'] = dict(k=1920, r0=0.38)

In [44]:
# ── Integrator ────────────────────────────────────────────────────────────────
langevin = hoomd.md.methods.Langevin(filter=hoomd.filter.All(), kT=KT)
sim.operations.integrator = hoomd.md.Integrator(
    dt=DT,
    methods=[langevin],
    forces=[lj1, lj2, yukawa, harmonic]
)

# ── Writer ────────────────────────────────────────────────────────────────────
gsd_writer = hoomd.write.GSD(
    filename=GSD_FILE,
    trigger=hoomd.trigger.Periodic(SAVE_EVERY),
    mode='wb'
)
sim.operations.writers.append(gsd_writer)



In [45]:
print(f"Running HPS simulation for {N_STEPS} steps...")
print(f"Sequence: {sequence} ({N} residues)")
print(f"Saving every {SAVE_EVERY} steps -> {N_STEPS // SAVE_EVERY} frames to {GSD_FILE}\n")

sim.run(N_STEPS)
# Force flush to disk
gsd_writer.flush()
print("Wrote", GSD_FILE)
print("Simulation complete!")


Running HPS simulation for 4082279 steps...
Sequence: GSHCFLDGIDKAQEEHEKYHSNWRAMASDFNLPPVVAKEIVASCDKCQLKGEAMHGQVDC (60 residues)
Saving every 1000 steps -> 4082 frames to very_cold_run.gsd

Wrote very_cold_run.gsd
Simulation complete!
